In [1]:
# Setup (Imports, API Key, Prompts)

import os
import json
import re
from openai import OpenAI
from tqdm.notebook import tqdm # ใช้ tqdm.notebook สำหรับ .ipynb
import pandas as pd
import time
from collections import defaultdict

print("🚀 Initializing AI Evaluator...")

# --- 1. Setup OpenAI Client ---
# <<< ⭐️⭐️⭐️ ไดจิต้องใส่ Key ตรงนี้ ⭐️⭐️⭐️ >>>
os.environ['OPENAI_API_KEY'] = "PUT_YOUR_KEY_HERE" 

if "OPENAI_API_KEY" not in os.environ or not os.environ['OPENAI_API_KEY'].startswith('sk-'):
    print("="*50)
    print("Error: OPENAI_API_KEY not set correctly.")
    print("="*50)
else:
    print("OpenAI API Key loaded.")
    
client = OpenAI()
EVALUATION_MODEL = "gpt-4o" # ใช้ "gpt-4o" ตัวเต็มเพื่อการประเมินที่แม่นยำ
print(f"Evaluation Model set to: {EVALUATION_MODEL}")

# --- 2. Define File Paths (Relative to this notebook) ---
OUR_MODEL_FILE = "../llms_therapist/therapist_simulation_results.jsonl"
BASELINE_MODEL_FILE = "../llms_therapist_base/therapist_simulation_results_BASELINE.jsonl"
print(f"Our Model results path: {OUR_MODEL_FILE}")
print(f"Baseline Model results path: {BASELINE_MODEL_FILE}")

# --- 3. Define Evaluation Prompts (Exactly as user provided) ---
SYSTEM_PROMPT = """You are an expert psychological evaluator specializing in Cognitive Behavioral Therapy (CBT) and therapeutic alliance. Your task is to analyze a full counseling session transcript and score the therapist's performance based on the evaluation frameworks used in the MIRROR paper (arXiv:2504.13211v2).

You must evaluate the entire session as a whole, not just individual turns.
"""

USER_PROMPT = """Please evaluate the following therapy session.

[Session Transcript]
{CONVERSATION_TEXT_GOES_HERE}
[/Session Transcript]

[Evaluation Task]
Analyze the therapist's responses throughout the session and provide scores for the following dimensions. Please be objective and critical.

1.  **Therapist Skills Assessment (General Counseling):**
    * **Understanding (Score 0-6):** How well did the therapist interpret the client's concerns and feelings? (0 = Not at all, 6 = Perfectly)
    * **Interpersonal Effectiveness (Score 0-6):** How well did the therapist maintain a supportive and therapeutic relationship? (0 = Not at all, 6 = Perfectly)

2.  **Client Alliance Assessment:**
    * **Affective Bond (Score 1-5):** How well did the therapist foster an emotional connection, trust, and empathy? (1 = Very Poor, 5 = Very Strong)

Please provide your scores in a clear JSON format:

{{  
  "therapist_skills": {{  
    "understanding": 0.0,
    "interpersonal_effectiveness": 0.0
  }},  
  "client_alliance": {{  
    "affective_bond": 0.0
  }},  
  "reasoning": "Provide a brief (1-2 sentence) justification for your scores here."
}}  
"""
print("✅ Setup complete. Prompts and paths are defined.")

🚀 Initializing AI Evaluator...
OpenAI API Key loaded.
Evaluation Model set to: gpt-4o
Our Model results path: ../llms_therapist/therapist_simulation_results.jsonl
Baseline Model results path: ../llms_therapist_base/therapist_simulation_results_BASELINE.jsonl
✅ Setup complete. Prompts and paths are defined.


In [2]:
# Data Loading & Formatting Functions

def get_sort_keys(filename):
    """Extracts (dialogue_num, utterance_num) as integers for sorting."""
    match = re.search(r'dialogue_(\d+)_utterance_(\d+)\.wav', filename)
    if match:
        dialogue_num = int(match.group(1))
        utterance_num = int(match.group(2))
        return (dialogue_num, utterance_num)
    else:
        return (999, 999) # Fallback for non-matching names

def load_and_group_results(filepath):
    """Loads a .jsonl file and groups it by dialogue_id."""
    if not os.path.exists(filepath):
        print(f"❌ ERROR: File not found at {filepath}")
        return None
        
    dialogues = defaultdict(list)
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line.strip())
                    dialogue_num, utterance_num = get_sort_keys(data['file_name'])
                    if dialogue_num != 999:
                        data['utterance_num'] = utterance_num # Store utterance number
                        dialogues[dialogue_num].append(data)
                except json.JSONDecodeError:
                    print(f"Warning: Skipping malformed JSON line in {filepath}")
    except Exception as e:
        print(f"❌ ERROR: Failed to read file {filepath}. Error: {e}")
        return None

    # Sort utterances within each dialogue
    sorted_dialogues = {}
    for dialogue_id, turns in dialogues.items():
        sorted_dialogues[dialogue_id] = sorted(turns, key=lambda x: x['utterance_num'])
        
    print(f"✅ Loaded and grouped {len(sorted_dialogues)} dialogues from {filepath}")
    return sorted_dialogues

def format_conversation_text(turns_list, is_baseline=False):
    """Converts a list of turns into a single 'Client: ... Therapist: ...' string."""
    full_text = ""
    therapist_key = "therapist_response_baseline" if is_baseline else "therapist_response"
    
    for turn in turns_list:
        full_text += f"CLIENT: {turn.get('transcript', '[missing transcript]')}\n\n"
        full_text += f"THERAPIST: {turn.get(therapist_key, '[missing response]')}\n\n"
        
    return full_text.strip()

print("✅ Data loading and formatting functions are defined.")

✅ Data loading and formatting functions are defined.


In [3]:
# Evaluation Function (Calling GPT-4o)

def evaluate_session(session_text, max_retries=3):
    """
    Calls the GPT-4o API to get evaluation scores.
    Retries on failure.
    """
    formatted_user_prompt = USER_PROMPT.format(CONVERSATION_TEXT_GOES_HERE=session_text)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": formatted_user_prompt}
    ]
    
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=EVALUATION_MODEL,
                messages=messages,
                temperature=0.0, # Crucial for objective scoring
                response_format={"type": "json_object"} # Force JSON output
            )
            
            response_content = completion.choices[0].message.content
            # Try to parse the JSON to ensure it's valid
            scores = json.loads(response_content)
            return scores # Success!
        
        except Exception as e:
            print(f"Warning: Attempt {attempt + 1}/{max_retries} failed. Error: {e}")
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
            else:
                return {
                    "error": f"Failed after {max_retries} retries.",
                    "last_error": str(e)
                }
    return None # Should not be reached

print("✅ AI Evaluation function is defined.")

✅ AI Evaluation function is defined.


In [4]:
# Main Execution Loop

print("--- Starting Evaluation Process ---")

# 1. Load both datasets
our_model_dialogues = load_and_group_results('../llms_therapist/therapist_simulation_results.jsonl')
baseline_dialogues = load_and_group_results('../llms_therapist_base/therapist_simulation_results_BASELINE.jsonl')

# Store results here
our_model_eval_scores = []
baseline_eval_scores = []

# Assuming dialogues 1-20 exist
num_dialogues = 20
if not our_model_dialogues or not baseline_dialogues:
    print("❌ ERROR: Cannot start evaluation. One or both input files failed to load.")
else:
    print(f"Found {len(our_model_dialogues)} 'Our Model' dialogues and {len(baseline_dialogues)} 'Baseline' dialogues.")
    
    for dialogue_id in tqdm(range(1, num_dialogues + 1), desc="Evaluating Dialogues"):
        
        # --- Evaluate Our Model ---
        if dialogue_id in our_model_dialogues:
            print(f"\nEvaluating OUR MODEL Dialogue {dialogue_id}...")
            session_text = format_conversation_text(our_model_dialogues[dialogue_id], is_baseline=False)
            scores = evaluate_session(session_text)
            scores['dialogue_id'] = dialogue_id # Add ID for reference
            our_model_eval_scores.append(scores)
            print(f"  -> Done. Score (Understanding): {scores.get('therapist_skills', {}).get('understanding')}")
        else:
            print(f"\nWarning: Skipping OUR MODEL Dialogue {dialogue_id} (not found in loaded data).")

        # --- Evaluate Baseline Model ---
        if dialogue_id in baseline_dialogues:
            print(f"Evaluating BASELINE Dialogue {dialogue_id}...")
            session_text = format_conversation_text(baseline_dialogues[dialogue_id], is_baseline=True)
            scores = evaluate_session(session_text)
            scores['dialogue_id'] = dialogue_id # Add ID for reference
            baseline_eval_scores.append(scores)
            print(f"  -> Done. Score (Understanding): {scores.get('therapist_skills', {}).get('understanding')}")
        else:
            print(f"\nWarning: Skipping BASELINE Dialogue {dialogue_id} (not found in loaded data).")

    print("\n🎉 --- Evaluation Process Complete! ---")

--- Starting Evaluation Process ---
✅ Loaded and grouped 20 dialogues from ../llms_therapist/therapist_simulation_results.jsonl
✅ Loaded and grouped 20 dialogues from ../llms_therapist_base/therapist_simulation_results_BASELINE.jsonl
Found 20 'Our Model' dialogues and 20 'Baseline' dialogues.


Evaluating Dialogues:   0%|          | 0/20 [00:00<?, ?it/s]


Evaluating OUR MODEL Dialogue 1...
  -> Done. Score (Understanding): 5.0
Evaluating BASELINE Dialogue 1...
  -> Done. Score (Understanding): 4.5

Evaluating OUR MODEL Dialogue 2...
  -> Done. Score (Understanding): 4.0
Evaluating BASELINE Dialogue 2...
  -> Done. Score (Understanding): 4.0

Evaluating OUR MODEL Dialogue 3...
  -> Done. Score (Understanding): 5.0
Evaluating BASELINE Dialogue 3...
  -> Done. Score (Understanding): 4.0

Evaluating OUR MODEL Dialogue 4...
  -> Done. Score (Understanding): 5.0
Evaluating BASELINE Dialogue 4...
  -> Done. Score (Understanding): 5.0

Evaluating OUR MODEL Dialogue 5...
  -> Done. Score (Understanding): 5.0
Evaluating BASELINE Dialogue 5...
  -> Done. Score (Understanding): 5.0

Evaluating OUR MODEL Dialogue 6...
  -> Done. Score (Understanding): 5.0
Evaluating BASELINE Dialogue 6...
  -> Done. Score (Understanding): 5.0

Evaluating OUR MODEL Dialogue 7...
  -> Done. Score (Understanding): 5.0
Evaluating BASELINE Dialogue 7...
  -> Done. Score

In [5]:
# Save Results

# Define output filenames
OUR_MODEL_SCORE_FILE = "ai_evaluation_results_OUR_MODEL.jsonl"
BASELINE_MODEL_SCORE_FILE = "ai_evaluation_results_BASELINE.jsonl"

# --- Save Our Model Scores ---
try:
    with open(OUR_MODEL_SCORE_FILE, 'w', encoding='utf-8') as f:
        for entry in our_model_eval_scores:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f"✅ Successfully saved 'Our Model' evaluation scores to '{OUR_MODEL_SCORE_FILE}'")
except Exception as e:
    print(f"❌ Error saving 'Our Model' scores: {e}")

# --- Save Baseline Model Scores ---
try:
    with open(BASELINE_MODEL_SCORE_FILE, 'w', encoding='utf-8') as f:
        for entry in baseline_eval_scores:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print(f"✅ Successfully saved 'Baseline Model' evaluation scores to '{BASELINE_MODEL_SCORE_FILE}'")
except Exception as e:
    print(f"❌ Error saving 'Baseline Model' scores: {e}")

✅ Successfully saved 'Our Model' evaluation scores to 'ai_evaluation_results_OUR_MODEL.jsonl'
✅ Successfully saved 'Baseline Model' evaluation scores to 'ai_evaluation_results_BASELINE.jsonl'


In [7]:
# Score Comparisons

import numpy as np

# --- Function to process scores and calculate averages ---
def calculate_averages(score_list):
    # Use pandas to easily flatten the nested JSON scores
    df = pd.json_normalize(score_list)
    
    # (ใหม่!) Filter out error rows, ONLY if the 'error' column exists
    if 'error' in df.columns:
        df = df[df['error'].isnull()]
    
    if df.empty:
        return {
            "understanding_avg": 0,
            "interpersonal_effectiveness_avg": 0,
            "affective_bond_avg": 0,
            "count": 0
        }
    
    # Calculate means
    averages = {
        "understanding_avg": df['therapist_skills.understanding'].mean(),
        "interpersonal_effectiveness_avg": df['therapist_skills.interpersonal_effectiveness'].mean(),
        "affective_bond_avg": df['client_alliance.affective_bond'].mean(),
        "count": len(df)
    }
    return averages

print("--- 📊 Final Score Comparison ---")

# Calculate averages for both lists
our_avg = calculate_averages(our_model_eval_scores)
base_avg = calculate_averages(baseline_eval_scores)

# Create a comparison DataFrame
summary_data = {
    "Metric": [
        "Understanding (0-6)", 
        "Interpersonal Effectiveness (0-6)", 
        "Affective Bond (1-5)",
        "Successful Dialogues"
    ],
    "Our Model (Vocal Context)": [
        f"{our_avg['understanding_avg']:.2f}",
        f"{our_avg['interpersonal_effectiveness_avg']:.2f}",
        f"{our_avg['affective_bond_avg']:.2f}",
        f"{our_avg['count']} / {num_dialogues}"
    ],
    "Baseline (Text-Only)": [
        f"{base_avg['understanding_avg']:.2f}",
        f"{base_avg['interpersonal_effectiveness_avg']:.2f}",
        f"{base_avg['affective_bond_avg']:.2f}",
        f"{base_avg['count']} / {num_dialogues}"
    ]
}

summary_df = pd.DataFrame(summary_data)

# Display the final comparison table
print("Average Scores Across All Evaluated Dialogues:")
display(summary_df)

--- 📊 Final Score Comparison ---
Average Scores Across All Evaluated Dialogues:


,Metric,Our Model (Vocal Context),Baseline (Text-Only)
0,Understanding (0-6),4.80,4.78
1,Interpersonal Effectiveness (0-6),4.78,4.75
2,Affective Bond (1-5),3.92,3.95
3,Successful Dialogues,20 / 20,20 / 20
